## 04 — SARIMAX Model
Dự báo số lượng bán cho Top 5 màu bằng SARIMAX (`auto_arima`) — 3 tháng Q2/2026.

Kiểm định ADF để xác nhận tính dừng.

In [ ]:
# Chuỗi thời gian theo màu
top5_colors  = top_colors.head(5).index.tolist()
months_q2    = pd.date_range("2026-04-01", periods=3, freq="MS")
months_label = ["T4/2026", "T5/2026", "T6/2026"]

color_monthly = (
    df[df["color"].isin(top5_colors)]
    .groupby(["month", "color"])["quantity"]
    .sum()
    .unstack(fill_value=0)
)
color_monthly.index = color_monthly.index.to_timestamp()

print("Top 5 màu:", top5_colors)
print("Shape:", color_monthly.shape)

In [ ]:
# Kiểm định ADF
print("Kiểm định ADF:")
for col in top5_colors:
    pv = adfuller(color_monthly[col].dropna())[1]
    print(f"  {col:<25}: p = {pv:.4f}  → {'DỪNG' if pv < 0.05 else 'KHÔNG DỪNG'}") 

In [ ]:
# Huấn luyện SARIMAX cho từng màu
forecasts_color = {}
for col in top5_colors:
    try:
        model_c = pm.auto_arima(color_monthly[col].values, seasonal=True, m=3,
                                stepwise=True, suppress_warnings=True, error_action="ignore")
        pred = model_c.predict(n_periods=3)
        forecasts_color[col] = np.maximum(pred, 0)
        print(f"  {col:<25}: {[int(v) for v in forecasts_color[col]]}")
    except Exception as e:
        forecasts_color[col] = np.zeros(3)
        print(f"  {col}: Lỗi — {e}")